# 분기 예정 시작회차 데이터셋
종로구·마포구 151개 상권, 외식업 10종, 2023~2025년 12분기를 유지한다. 결과를 보기 전에 고정한 `revised_quarterly_plan.json`에 따라 공연 **예정 시작회차**를 집계한다. 실제 상연회차·관객·공연 직후 매출로 해석하지 않는다.

기본 공간은 상권 중심점–공연시설 500m, 비교 공간은 기존 상권 폴리곤–공연시설 최단거리 500m다. 가용률 분모는 파싱 전 공식 연결 후보 ID이며, 분기 내 정상요일 예정시작이 하나 이상 생성되는 후보만 가용으로 센다. 러닝타임 결측이나 비정상 길이 자체는 제외 사유가 아니다. 단 러닝타임이 정상적으로 파싱되면서 같은 ID의 예상 상연시간이 겹치는 경우는 보수적으로 제외한다.

매출 원값과 결측을 보존하며, 회귀 적합·p값 확인·네트워크 요청 없이 데이터와 품질표만 만든다.

## 1. 고정 계획과 입력 확인
기존 최종 패널·공식 연결·KOPIS 스냅샷·서울 경계의 해시를 검증한다. 이전 분석 노트북과 지정된 보호 파일도 실행 전후 해시가 같은지 확인한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime,timedelta
import hashlib,json,re
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display,Markdown
ROOT=PROJECT_ROOT
DATA=project_path('dataset/analysis/final/revised_quarterly')
REPORTS=project_path('reports/20261006/revised_quarterly')
DATA.mkdir(parents=True,exist_ok=True);REPORTS.mkdir(parents=True,exist_ok=True)
inputs={};checks=[]
def check(name,passed):
    checks.append(dict(check=name,passed=bool(passed)))
    assert passed,name

def register(relative,expected=None):
    path=project_path(relative);digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None:check('hash:'+relative,digest==expected)
    if relative not in inputs:inputs[relative]=dict(path=relative,sha256_before=digest,expected_sha256=expected)
    elif expected is not None:inputs[relative]['expected_sha256']=expected
    return path

def read_json(relative):return project_json_loads(register(relative).read_text(encoding='utf-8'))
def read_csv(relative,expected=None,**kwargs):return pd.read_csv(register(relative,expected),float_precision='round_trip',**kwargs)
PLAN_PATH='jupyter-notebook/20261006/metadata/revised_quarterly_plan.json'
plan=read_json(PLAN_PATH)
check('plan_primary_space',plan['primary_spatial_definition']=='centroid500')
check('plan_missing_runtime_kept',plan['missing_runtime_excluded'] is False)
check('plan_invalid_runtime_kept',plan['runtime_over6h_excluded_by_itself'] is False)
for relative,digest in plan['protected_source_sha256'].items():register(relative,digest)
for path in sorted((project_path('jupyter-notebook/20261006/notebooks')).glob('*.ipynb')):
    if path.name[:2].isdigit() and int(path.name[:2])<12:register(path.relative_to(ROOT).as_posix())

In [ ]:
scope=read_json('reports/20261006/08_food_scope_metadata.json')
kopis=read_json('jupyter-notebook/20261006/metadata/kopis_analysis_sources.json')
seoul=read_json('jupyter-notebook/20261006/metadata/seoul_analysis_sources.json')
check('kopis_validated',kopis['validated'])
expected={r['path']:r['sha256'] for r in scope['source_files']}
key_types={'area_id':str,'mt20id':str,'mt10id':str,'quarter':str,'industry_id':str}
PANEL_PATH='dataset/analysis/final/food_industry_panel.csv'
panel=read_csv(PANEL_PATH,scope['panel_sha256'],dtype=key_types)
links=read_csv('dataset/analysis/final/performance_facility_quarter.csv',expected['dataset/analysis/final/performance_facility_quarter.csv'],dtype=key_types)
areas=read_csv('dataset/analysis/final/study_areas.csv',expected['dataset/analysis/final/study_areas.csv'],dtype=key_types)
old_distances=read_csv('dataset/analysis/final/venue_area_distances_800m.csv',expected['dataset/analysis/final/venue_area_distances_800m.csv'],dtype=key_types)
frames={}
for role in ['performance_details','facility_details']:
    spec=kopis['files'][role]
    frames[role]=read_csv(spec['path'],spec['sha256'],dtype=str,keep_default_na=False)
performances,facilities=frames['performance_details'],frames['facility_details']
QUARTERS=sorted(panel.quarter.unique());SPACES=['centroid500','min_distance500']
official_links=links.loc[links.link_method.eq('official_detail')].copy()
check('source_key_unique',not official_links.duplicated(['mt20id','quarter']).any())
check('panel_key_unique',not panel.duplicated(['area_id','industry_id','quarter']).any())
check('panel_scope',len(panel)==18120 and len(areas)==151 and len(QUARTERS)==12)
check('all_official_details_present',set(official_links.mt20id).issubset(set(performances.mt20id)))
check('official_facility_ids_agree',official_links.merge(performances[['mt20id','mt10id']],on='mt20id',suffixes=('_link','_detail'),validate='many_to_one').eval('mt10id_link == mt10id_detail').all())

**해석:** 시설명 임시 연결은 이 데이터셋의 후보에 넣지 않는다. 공식 시설 ID 및 고정 현재 좌표로 연결 가능한 등록 공연 범위의 자료다. 좌표가 없거나 과거와 위치가 달랐던 시설의 노출을 복원하는 자료는 아니다.

## 2. 두 공간 기준을 독립 계산
EPSG:5181에서 폴리곤 중심점과 시설점의 유클리드 거리를 계산한다. 최단거리 기준도 같은 고정 폴리곤에서 다시 계산하여 기존 거리와 대조한다. 중심점이 폴리곤 밖에 있는 상권을 표시하며, 중심점 기준도 전체 상권 매출의 근사 연결이라는 한계가 남는다.

In [ ]:
import shapefile,shapely
from shapely.geometry import shape,Point
from pyproj import CRS,Transformer
area_source=next(item for item in seoul['items'] if item['dataset']=='areas')
for entry in area_source['extracted']:register(entry['path'],entry['sha256'])
shp=project_path(next(entry['path'] for entry in area_source['extracted'] if entry['path'].endswith('.shp')))
check('projected_CRS_5181',CRS.from_wkt(shp.with_suffix('.prj').read_text()).to_epsg()==5181)
reader=shapefile.Reader(str(shp),encoding=shp.with_suffix('.cpg').read_text().strip())
geometry={};area_ids=set(areas.area_id)
for item in reader.iterShapeRecords():
    area_id=str(item.record.as_dict()['TRDAR_CD']).removesuffix('.0')
    if area_id not in area_ids:continue
    g=shape(item.shape.__geo_interface__)
    if not g.is_valid:
        g=shapely.make_valid(g)
        if g.geom_type=='GeometryCollection':g=shapely.union_all([part for part in g.geoms if part.geom_type in ['Polygon','MultiPolygon']])
    check('valid_polygon:'+area_id,g.is_valid and not g.is_empty and g.area>0)
    geometry[area_id]=g
centroids=areas.copy()
centroids['centroid_x_m']=centroids.area_id.map(lambda key:geometry[key].centroid.x)
centroids['centroid_y_m']=centroids.area_id.map(lambda key:geometry[key].centroid.y)
centroids['centroid_inside_polygon']=centroids.area_id.map(lambda key:geometry[key].covers(geometry[key].centroid))
centroids['polygon_area_km2']=centroids.area_id.map(lambda key:geometry[key].area/1e6)
centroids['crs_epsg']=5181
check('all_areas_have_geometry',set(geometry)==area_ids)
display(centroids.loc[~centroids.centroid_inside_polygon,['area_id','area_name','district','centroid_inside_polygon']])

In [ ]:
candidate_facilities=facilities.loc[facilities.mt10id.isin(official_links.mt10id)].copy()
lat=pd.to_numeric(candidate_facilities.la,errors='coerce');lon=pd.to_numeric(candidate_facilities.lo,errors='coerce')
valid_coord=lat.between(33,39)&lon.between(124,132)
coordinate_quality={'official_facility_ids':int(official_links.mt10id.nunique()),
    'facility_rows_available':len(candidate_facilities),'valid_current_coordinates':int(valid_coord.sum()),
    'missing_or_invalid_current_coordinates':int((~valid_coord).sum())}
located=candidate_facilities.loc[valid_coord].copy()
xs,ys=Transformer.from_crs(4326,5181,always_xy=True).transform(lon[valid_coord],lat[valid_coord])
points=np.array([Point(x,y) for x,y in zip(xs,ys)],dtype=object)
spatial_rows=[]
for row in centroids.itertuples(index=False):
    d_centroid=np.hypot(xs-row.centroid_x_m,ys-row.centroid_y_m)
    d_min=shapely.distance(geometry[row.area_id],points)
    for space,selected in [('centroid500',d_centroid<=500),('min_distance500',d_min<=500)]:
        for idx in np.flatnonzero(selected):
            spatial_rows.append(dict(area_id=row.area_id,mt10id=located.iloc[idx].mt10id,
                spatial_definition=space,centroid_distance_m=float(d_centroid[idx]),min_distance_m=float(d_min[idx])))
spatial_links=pd.DataFrame(spatial_rows).sort_values(['spatial_definition','area_id','mt10id']).reset_index(drop=True)
check('spatial_links_unique',not spatial_links.duplicated(['spatial_definition','area_id','mt10id']).any())
old_min=old_distances.loc[old_distances.distance_m.le(500)&old_distances.mt10id.isin(official_links.mt10id),['area_id','mt10id','distance_m']]
new_min=spatial_links.loc[spatial_links.spatial_definition.eq('min_distance500')]
compare=new_min.merge(old_min,on=['area_id','mt10id'],how='outer',indicator=True,validate='one_to_one')
check('min500_pairs_match_old',compare['_merge'].eq('both').all())
check('min500_distances_match_old',np.allclose(compare.min_distance_m,compare.distance_m,atol=1e-5,rtol=0))
display(spatial_links.groupby('spatial_definition').agg(facility_area_pairs=('mt10id','size'),facilities=('mt10id','nunique'),areas=('area_id','nunique')))

**해석:** 가까운 중심점은 공연장을 가까운 음식점과 직접 연결한 결과가 아니다. 원 매출의 공간 단위는 상권 전체다. 폴리곤 면적이나 중심점 거리를 이용해 매출을 임의로 나누지 않는다.

## 3. 기존 엄격 일정 파서 재사용
09번 노트북의 요일·시각 파서와 러닝타임·겹침 점검 함수를 동일하게 복사하여 검증한다. HOL 전용 일정은 일반 요일 시작을 만들 수 없으므로 미가용이며, HOL과 일반 요일이 함께 있으면 일반 요일만 반복한다. 공휴일 달력을 추정하지 않는다.

In [ ]:
DAY_LABELS = ['월요일','화요일','수요일','목요일','금요일','토요일','일요일']
DAY_INDEX = {name:i for i,name in enumerate(DAY_LABELS)}
DAY_RX = '|'.join(DAY_LABELS)
BLOCK = re.compile(rf'({DAY_RX}|HOL)(?:\s*~\s*({DAY_RX}))?\(([^()]*)\)')
CLOCK = re.compile(r'(?:[01]?\d|2[0-3]):[0-5]\d')
def parse_schedule(text):
    text = str(text).strip()
    result = dict(schedule_status='ok',has_hol='HOL' in text,weekday_times={})
    if not text:
        result['schedule_status']='missing_schedule'; return result
    matches = list(BLOCK.finditer(text))
    if not matches or BLOCK.sub('',text).replace(',','').strip():
        result['schedule_status']='unsupported_schedule'; return result
    if (text[:matches[0].start()].strip() or text[matches[-1].end():].strip()
        or any(re.fullmatch(r'\s*,\s*', text[a.end():b.start()]) is None
               for a,b in zip(matches[:-1],matches[1:]))):
        result['schedule_status']='unsupported_schedule'; return result
    for match in matches:
        first,last,clocks = match.groups()
        if first=='HOL' and last is not None:
            result['schedule_status']='unsupported_schedule'; return result
        tokens = [token.strip() for token in clocks.split(',')]
        if not tokens or not all(CLOCK.fullmatch(token) for token in tokens):
            result['schedule_status']='invalid_clock'; return result
        times = [int(token.split(':')[0])*60+int(token.split(':')[1]) for token in tokens]
        if len(set(times)) != len(times):
            result['schedule_status']='duplicate_start_time'; return result
        if first=='HOL':
            continue
        left,right=DAY_INDEX[first],DAY_INDEX[last or first]
        if left>right:
            result['schedule_status']='reverse_weekday_range'; return result
        for day in range(left,right+1):
            if day in result['weekday_times']:
                result['schedule_status']='overlapping_weekday_blocks'; return result
            result['weekday_times'][day]=sorted(times)
    if not result['weekday_times']:
        result['schedule_status']='hol_only_no_normal_schedule'
    return result

In [ ]:
def parse_runtime(text):
    text=str(text).strip()
    if not text:
        return np.nan,'missing_runtime'
    match=re.fullmatch(r'(?:(\d+)시간)?\s*(?:(\d+)분)?',text)
    if not match or not any(match.groups()):
        return np.nan,'unsupported_runtime'
    hours,minutes=match.groups()
    if hours and minutes and int(minutes)>=60:
        return np.nan,'noncanonical_runtime'
    value=60*int(hours or 0)+int(minutes or 0)
    if value<=0: return np.nan,'nonpositive_runtime'
    if value>360: return float(value),'runtime_over_6h'
    return float(value),'ok'
def overlaps_own_runtime(weekday_times, runtime):
    if not weekday_times or not np.isfinite(runtime): return False
    starts=sorted(day*1440+time for day,times in weekday_times.items() for time in times)
    gaps=np.diff(starts+[starts[0]+7*1440])
    return bool(np.any(gaps<runtime))

In [ ]:
source09=nbformat.read(project_path('jupyter-notebook/20261006/notebooks/09_공연시간대_노출구축.ipynb'),as_version=4)
source_parser_hashes={str(i):hashlib.sha256(source09.cells[i].source.encode()).hexdigest() for i in [6,7]}
check('schedule_parser_copy_exact',source_parser_hashes['6']=='d90fa147560e60590cc6ef9d8e2e4d733689e650adb07fad26edf4ee741e2b13')
check('runtime_parser_copy_exact',source_parser_hashes['7']=='69b4ad10899808d44c21556a6a90bac7a1684d5f40162e79473e9d1150e9be25')

In [ ]:
edge_rows=[]
for text,expected_status in [
    ('월요일 ~ 금요일(19:30), 토요일(15:00,19:00), HOL(14:00)','ok'),
    ('월요일(00:30)','ok'),('HOL(14:00)','hol_only_no_normal_schedule'),
    ('월요일(19:00,19:00)','duplicate_start_time'),
    ('월요일(19:00), 월요일(20:00)','overlapping_weekday_blocks'),
    ('금요일 ~ 월요일(19:00)','reverse_weekday_range'),
    ('월요일(24:00)','invalid_clock'),('매일(19:00)','unsupported_schedule'),
    ('월요일(19:00),,화요일(20:00)','unsupported_schedule'),
    ('HOL ~ 월요일(19:00)','unsupported_schedule')]:
    actual=parse_schedule(text)['schedule_status']
    edge_rows.append(dict(test='schedule',input=text,expected=expected_status,actual=actual,passed=actual==expected_status))
for text,expected_status in [('1시간 30분','ok'),('85분','ok'),('1시간 60분','noncanonical_runtime'),('7시간','runtime_over_6h')]:
    actual=parse_runtime(text)[1]
    edge_rows.append(dict(test='runtime',input=text,expected=expected_status,actual=actual,passed=actual==expected_status))
assert overlaps_own_runtime({6:[23*60+30],0:[30]},90)
assert not overlaps_own_runtime({0:[19*60],1:[19*60]},90)
assert all(row['passed'] for row in edge_rows)
check('manual_parser_edges',all(row['passed'] for row in edge_rows))
display(pd.DataFrame(edge_rows))

## 4. 후보 ID의 시작일정 품질
가용 여부를 판정하기 전에 공간별 전체 공식 ID×분기 후보를 만든다. 러닝타임은 시작 횟수를 만드는 데 사용하지 않고 알려진 겹침 모호성 제외와 품질표에만 사용한다. 정상 러닝타임이 없는 ID는 시작일정이 유효하면 가용할 수 있다.

In [ ]:
candidate_relations=official_links.merge(spatial_links,on='mt10id',how='inner',validate='many_to_many')
candidate_quarters=candidate_relations[['mt20id','mt10id','quarter']].drop_duplicates().sort_values(['mt20id','quarter'])
candidate_ids=set(candidate_quarters.mt20id)
check('candidate_ID_quarter_unique',not candidate_quarters.duplicated(['mt20id','quarter']).any())
audit_rows=[];schedules={}
for row in performances.loc[performances.mt20id.isin(candidate_ids)].itertuples(index=False):
    parsed=parse_schedule(row.dtguidance)
    runtime,runtime_status=parse_runtime(row.prfruntime)
    start=pd.to_datetime(row.prfpdfrom,format='%Y.%m.%d',errors='coerce')
    stop=pd.to_datetime(row.prfpdto,format='%Y.%m.%d',errors='coerce')
    valid=bool(pd.notna(start) and pd.notna(stop) and start<=stop)
    overlap=overlaps_own_runtime(parsed['weekday_times'],runtime)
    excluded_overlap=runtime_status=='ok' and overlap
    schedules[row.mt20id]=parsed['weekday_times']
    audit_rows.append(dict(mt20id=row.mt20id,mt10id=row.mt10id,dtguidance=row.dtguidance,
        prfruntime=row.prfruntime,prfpdfrom=row.prfpdfrom,prfpdto=row.prfpdto,
        schedule_status=parsed['schedule_status'],valid_period=valid,has_hol=parsed['has_hol'],
        runtime_status=runtime_status,runtime_minutes=runtime,own_runtime_overlap_ambiguous=overlap,
        excluded_known_valid_runtime_overlap=excluded_overlap,
        start_schedule_valid=parsed['schedule_status']=='ok' and valid and not excluded_overlap,
        period_extends_beyond_2025=bool(valid and stop>=pd.Timestamp('2026-01-01'))))
audit=pd.DataFrame(audit_rows)
check('all_candidates_audited',set(audit.mt20id)==candidate_ids)
display(audit.groupby(['schedule_status','runtime_status','excluded_known_valid_runtime_overlap']).size().rename('공연_ID수').reset_index())

## 5. 분기 내 예정 시작 횟수 생성
시작일시가 속한 분기에 한 번 귀속한다. 기존 분기 후보 안에서 등록기간과 분기기간의 교집합만 순회한다. 종료 후 창과 달리 인접분기 의존 후보나 연구기간 밖 패딩 회차가 필요하지 않다. 후보는 있으나 그 분기에 일반 요일 시작이 하나도 생성되지 않으면 미가용으로 표시하여 실제 0회 공연으로 단정하지 않는다.

In [ ]:
def quarter_bounds(quarter):
    start=pd.Timestamp(year=int(quarter[:4]),month=3*(int(quarter[-1])-1)+1,day=1)
    stop=start+pd.offsets.QuarterBegin(startingMonth=start.month)
    return start,stop

def scheduled_starts_in_quarter(weekday_times,registered_start,registered_stop,quarter):
    q_start,q_stop=quarter_bounds(quarter)
    first=max(registered_start,q_start);last=min(registered_stop,q_stop-pd.Timedelta(days=1))
    if first>last:return []
    return [day+pd.Timedelta(minutes=minutes) for day in pd.date_range(first,last,freq='D')
            for minutes in weekday_times.get(day.weekday(),[])]

for q in QUARTERS:
    begin,end=quarter_bounds(q)
    check('quarter_bound:'+q,begin.month in [1,4,7,10] and 89<=(end-begin).days<=92)
check('quarter_boundary_no_ghost_start',len(scheduled_starts_in_quarter({6:[600]},pd.Timestamp('2023-09-30'),pd.Timestamp('2023-10-01'),'20233'))==0)
check('quarter_boundary_start_assigned_once',len(scheduled_starts_in_quarter({6:[600]},pd.Timestamp('2023-09-30'),pd.Timestamp('2023-10-01'),'20234'))==1)
check('midnight_in_current_quarter',scheduled_starts_in_quarter({6:[0]},pd.Timestamp('2023-01-01'),pd.Timestamp('2023-01-01'),'20231')==[pd.Timestamp('2023-01-01')])

In [ ]:
qrows=[];all_generated_keys=[]
for row in candidate_quarters.merge(audit,on=['mt20id','mt10id'],validate='many_to_one').itertuples(index=False):
    starts=[];period_overlap=False;generated_count=np.nan
    if row.valid_period:
        begin=pd.to_datetime(row.prfpdfrom,format='%Y.%m.%d');end=pd.to_datetime(row.prfpdto,format='%Y.%m.%d')
        qb,qe=quarter_bounds(row.quarter);period_overlap=begin<qe and end>=qb
        if row.schedule_status=='ok':
            starts=scheduled_starts_in_quarter(schedules[row.mt20id],begin,end,row.quarter)
            generated_count=len(starts)
            all_generated_keys.extend((row.mt20id,t) for t in starts)
            check('starts_inside_source_quarter:'+row.mt20id+':'+row.quarter,
                  all(qb<=t<qe and begin<=t<end+pd.Timedelta(days=1) for t in starts))
    eligible=bool(row.start_schedule_valid and len(starts)>0)
    qrows.append(dict(mt20id=row.mt20id,mt10id=row.mt10id,quarter=row.quarter,
        source_period_overlaps_quarter=bool(period_overlap),generated_start_count=generated_count,
        first_generated_start=starts[0].isoformat() if starts else None,
        last_generated_start=starts[-1].isoformat() if starts else None,
        eligible_id_quarter=eligible,known_scheduled_start_count=len(starts) if eligible else 0,
        no_matching_regular_start=bool(row.schedule_status=='ok' and row.valid_period and len(starts)==0),
        has_hol=row.has_hol,excluded_known_valid_runtime_overlap=row.excluded_known_valid_runtime_overlap,
        schedule_status=row.schedule_status,runtime_status=row.runtime_status,valid_period=row.valid_period,
        period_extends_beyond_2025=row.period_extends_beyond_2025))
performance_quarters=pd.DataFrame(qrows)
check('nominal_start_not_duplicated',len(all_generated_keys)==len(set(all_generated_keys)))
check('candidate_quarters_preserved',len(performance_quarters)==len(candidate_quarters))
check('eligible_requires_positive_starts',performance_quarters.loc[performance_quarters.eligible_id_quarter,'known_scheduled_start_count'].gt(0).all())
display(performance_quarters[['generated_start_count','eligible_id_quarter','known_scheduled_start_count','no_matching_regular_start']].describe(include='all'))

**해석:** 생성된 회차는 현재 상세 스냅샷의 요일·시각을 등록기간에 반복한 예정 횟수다. 실제 날짜별 예외, 휴연, 일정 변경 이력, 관객은 확인하지 못한다. 러닝타임 겹침 제외는 보수적인 품질 규칙이며 다른 모든 일정의 진위를 검증했다는 뜻이 아니다.

## 6. 상권×분기×공간 기준의 가용률과 노출
분모는 파싱 전 후보 ID 수, 분자는 해당 분기에 가용한 ID 수다. 가용률 100%일 때만 완전 노출 열에 값을 제공한다. `known_scheduled_start_count`는 미가용 ID를 제외한 부분 합계이며 90% 민감도에서만 제한적으로 사용할 수 있다. 가용률은 ID 기준으로서 실제 회차나 관객의 관측 비중이 아니다.

In [ ]:
rel=candidate_relations[['spatial_definition','area_id','quarter','mt20id','mt10id']].merge(
    performance_quarters,on=['mt20id','mt10id','quarter'],validate='many_to_one')
rel['schedule_unavailable']=rel.schedule_status.ne('ok')|~rel.valid_period
rel['runtime_missing']=rel.runtime_status.eq('missing_runtime')
rel['runtime_not_ok']=rel.runtime_status.ne('ok')
agg=rel.groupby(['area_id','quarter','spatial_definition']).agg(
    candidate_id_count=('mt20id','nunique'),eligible_id_count=('eligible_id_quarter','sum'),
    known_scheduled_start_count=('known_scheduled_start_count','sum'),
    schedule_unavailable_candidate_ids=('schedule_unavailable','sum'),
    no_matching_regular_start_candidate_ids=('no_matching_regular_start','sum'),
    excluded_known_valid_runtime_overlap_candidate_ids=('excluded_known_valid_runtime_overlap','sum'),
    has_hol_candidate_ids=('has_hol','sum'),runtime_missing_candidate_ids=('runtime_missing','sum'),
    runtime_not_ok_candidate_ids=('runtime_not_ok','sum'),
    period_extends_beyond_2025_candidate_ids=('period_extends_beyond_2025','sum'))
grid=pd.MultiIndex.from_product([sorted(areas.area_id),QUARTERS,SPACES],names=['area_id','quarter','spatial_definition'])
exposure=agg.reindex(grid,fill_value=0).reset_index()
integer_columns=list(agg.columns)
exposure[integer_columns]=exposure[integer_columns].astype('int64')
exposure['zero_registered_candidate']=exposure.candidate_id_count.eq(0)
exposure['availability_pct']=100*exposure.eligible_id_count/exposure.candidate_id_count.replace(0,np.nan)
exposure.loc[exposure.zero_registered_candidate,'availability_pct']=100.0
exposure['complete_schedule_coverage']=exposure.eligible_id_count.eq(exposure.candidate_id_count)
exposure['scheduled_start_count']=exposure.known_scheduled_start_count.where(exposure.complete_schedule_coverage)
check('exposure_all_keys',len(exposure)==151*12*2 and not exposure.duplicated(['area_id','quarter','spatial_definition']).any())
check('eligible_within_candidates',exposure.eligible_id_count.le(exposure.candidate_id_count).all())
check('partial_exposure_masked',exposure.loc[~exposure.complete_schedule_coverage,'scheduled_start_count'].isna().all())
check('complete_zero_means_no_registered_candidates',exposure.loc[exposure.complete_schedule_coverage & exposure.scheduled_start_count.eq(0),'zero_registered_candidate'].all())
display(exposure.groupby('spatial_definition').agg(area_quarters=('area_id','size'),complete=('complete_schedule_coverage','sum'),zero_registered_candidate=('zero_registered_candidate','sum')))

**노출 0의 의미:** 등록 후보가 없는 상권·분기는 가용률 100%, 예정 시작 0으로 둔다. 이는 고정된 공개등록·공식 연결·현재 좌표 범위에서 후보가 없다는 뜻이다. 미파싱 공연이나 실제 미등록 공연까지 존재하지 않았다는 뜻이 아니다.

## 7. 원 패널에 결합하고 관측키·매출 검증
151×10×12 관측키를 모두 유지한다. 중심점과 두 공간 노출을 열로 결합하여 13번 분석이 같은 원 매출을 사용할 수 있게 한다. 원 매출 결측과 실제 야간 0원을 그대로 보존한다.

In [ ]:
revised=panel.copy()
for space in SPACES:
    part=exposure.loc[exposure.spatial_definition.eq(space)].drop(columns='spatial_definition')
    part=part.rename(columns={name:f'{name}_{space}' for name in part.columns if name not in ['area_id','quarter']})
    revised=revised.merge(part,on=['area_id','quarter'],how='left',validate='many_to_one',sort=False)
revised=revised.merge(centroids[['area_id','centroid_x_m','centroid_y_m','centroid_inside_polygon','crs_epsg']],on='area_id',how='left',validate='many_to_one',sort=False)
pd.testing.assert_frame_equal(revised[panel.columns],panel,check_dtype=True,check_exact=True)
check('original_panel_columns_exact',True)
check('revised_keys_preserved',len(revised)==18120 and not revised.duplicated(['area_id','industry_id','quarter']).any())
check('sales_missingness_preserved',revised[['total_sales','night_sales']].isna().equals(panel[['total_sales','night_sales']].isna()))
check('centroid_coordinates_finite',np.isfinite(revised[['centroid_x_m','centroid_y_m']]).all().all())
sample_rows=[]
for space in SPACES:
    for threshold in [100,90]:
        aq=exposure.loc[exposure.spatial_definition.eq(space)&exposure.availability_pct.ge(threshold)]
        part=revised.loc[revised[f'availability_pct_{space}'].ge(threshold)]
        observed=part.loc[part.total_sales.notna()]
        common=part.loc[part.total_sales.gt(0)&part.night_sales.notna()&part.night_sales.ge(0)]
        n_per_entity=common.groupby(['area_id','industry_id']).size()
        repeat=common.set_index(['area_id','industry_id']).loc[lambda x:x.index.isin(n_per_entity[n_per_entity.ge(2)].index)].reset_index()
        sample_rows.append(dict(spatial_definition=space,availability_threshold_pct=threshold,
            area_quarters=len(aq),zero_candidate_area_quarters=int(aq.zero_registered_candidate.sum()),
            observed_total_sales_rows=len(observed),observed_total_sales_areas=int(observed.area_id.nunique()),
            common_outcome_rows=len(common),common_outcome_areas=int(common.area_id.nunique()),
            repeated_entity_rows=len(repeat),repeated_entity_areas=int(repeat.area_id.nunique())))
samples=pd.DataFrame(sample_rows)
display(samples)

**해석:** 표는 가용 표본의 구조만 보여준다. 반복 관측 수는 회귀의 최종 표본 수나 식별 강도를 보장하지 않는다. 고정효과, 업종별 최소 군집 수 등 분석 필터는 13번에서 별도로 적용한다. 노출과 매출의 방향이나 유의성을 보고 기준을 바꾸지 않는다.

## 8. 새 자료 저장·입력 불변 확인
기존 원본과 분석 결과는 수정하지 않는다. 재현에 필요한 공연 ID×분기 예정 시작 집계와 품질 감사표를 함께 저장한다. 일별 예정 시작 전체를 별도 대용량 파일로 남기지 않아도 이 노트북의 일정 파서와 생성 규칙으로 재현할 수 있다.

In [ ]:
data_outputs={
    'area_quarter_exposure.csv':exposure,'revised_quarterly_panel.csv':revised,
    'area_centroids.csv':centroids,'performance_quarter_schedule.csv':performance_quarters,
    'performance_schedule_audit.csv':audit,'facility_area_links.csv':spatial_links}
report_outputs={'12_sample_availability.csv':samples,'12_parser_edge_checks.csv':pd.DataFrame(edge_rows)}
output_records={}
for directory,mapping in [(DATA,data_outputs),(REPORTS,report_outputs)]:
    for name,frame in mapping.items():
        path=directory/name;frame.to_csv(path,index=False,encoding='utf-8-sig')
        output_records[path.relative_to(ROOT).as_posix()]={'rows':len(frame),'columns':frame.columns.tolist(),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
reloaded=pd.read_csv(DATA/'revised_quarterly_panel.csv',dtype=key_types,float_precision='round_trip')
pd.testing.assert_frame_equal(reloaded[panel.columns],panel,check_dtype=True,check_exact=True)
check('saved_original_values_exact',True)
for relative,record in inputs.items():
    record['sha256_after']=hashlib.sha256((project_path(relative)).read_bytes()).hexdigest()
    record['unchanged']=record['sha256_after']==record['sha256_before']
check('all_registered_inputs_unchanged',all(row['unchanged'] for row in inputs.values()))
validation=pd.DataFrame(checks)
path=REPORTS/'12_validation_checks.csv';validation.to_csv(path,index=False,encoding='utf-8-sig')
output_records[path.relative_to(ROOT).as_posix()]={'rows':len(validation),'columns':validation.columns.tolist(),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}

In [ ]:
panel_path=(DATA/'revised_quarterly_panel.csv').relative_to(ROOT).as_posix()
summary={'status':'complete','created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'plan_path':PLAN_PATH,'plan_sha256':inputs[PLAN_PATH]['sha256_before'],
    'panel_path':panel_path,'panel_sha256':output_records[panel_path]['sha256'],
    'all_inputs_unchanged':True,'validation_all_passed':bool(validation.passed.all()),
    'validation_checks':len(validation),'new_regression_fits':0,'network_requests':0,
    'scope':{'areas':len(areas),'industries':int(panel.industry_id.nunique()),'quarters':QUARTERS,'panel_rows':len(revised)},
    'spatial_definitions':SPACES,'radius_m':500,'coordinate_crs_epsg':5181,
    'centroids_outside_polygon':centroids.loc[~centroids.centroid_inside_polygon,'area_id'].tolist(),
    'coordinate_quality':coordinate_quality,'kopis_snapshot_run_id':kopis['run_id'],
    'candidate_performance_IDs':len(candidate_ids),'candidate_ID_quarters':len(performance_quarters),
    'eligible_ID_quarters':int(performance_quarters.eligible_id_quarter.sum()),
    'known_nominal_start_count_unique_ID_quarters':int(performance_quarters.known_scheduled_start_count.sum()),
    'source_quarters_not_overlapping_current_registered_period':int((~performance_quarters.source_period_overlaps_quarter).sum()),
    'quarter_candidates_without_regular_start':int(performance_quarters.no_matching_regular_start.sum()),
    'parser_source_notebook':'jupyter-notebook/20261006/notebooks/09_공연시간대_노출구축.ipynb','parser_source_cell_sha256':source_parser_hashes,
    'eligibility_rule':plan['availability_numerator'],'availability_denominator':plan['availability_denominator'],
    'zero_rule':'no observed official registered candidate -> 100pct coverage and zero nominal starts; not proof of no actual performance',
    'partial_sum_rule':'known_scheduled_start_count excludes unavailable candidates; scheduled_start_count masked unless complete',
    'runtime_role':'runtime_status ok AND own runtime overlap excluded; missing, unsupported, and over6h runtime not excluded by themselves',
    'holiday_assumption':plan['holiday_assumption'],'normal_weekday_recurrence_assumption':True,
    'actual_sessions_observed':False,'actual_attendance_observed':False,'actual_post_event_sales_observed':False,
    'historical_schedule_versions_verified':False,'timezone':'Asia/Seoul local naive scheduled datetime',
    'sales_missing_zero_filled':False,'sales_spatially_prorated':False,
    'sample_availability':samples.to_dict('records'),
    'limitations':['current schedule repeated into historical registered periods without date-specific revisions or cancellations',
        'HOL clauses not combined with a verified historical holiday calendar',
        'coverage measures candidate IDs, not true session or audience coverage',
        'current coordinates and boundaries applied to historical quarters; centroid can be outside polygon',
        'whole-area sales remain spatial aggregates; shared performances may induce spatial dependence',
        'quarterly total and fixed21-24h sales do not measure exact performance-following transactions'],
    'inputs':list(inputs.values()),'input_sha256':{p:r['sha256_before'] for p,r in inputs.items()},'outputs':output_records}
(REPORTS/'12_dataset_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'완료: 패널 {len(revised):,}행 / 공간별 노출 {len(exposure):,}행 / 검증 {len(validation):,}개 통과')
print(f'입력 {len(inputs)}개 불변, 새 회귀 0, 네트워크 요청 0')

**완료 후 사용:** 주 분석은 중심점 500m·가용률 100%의 예정 시작회차와 총매출액을 사용한다. 90% 부분 합계, 최단거리 500m, 같은 공간·표본의 공연 ID 수 비교는 고정 계획에 따라 13번에서 수행한다. 이번 자료는 실제 공연 회차나 관객 노출을 새로 확보한 결과가 아니다.